# 🎯 Day 31 — Mock Interview Practice — **FAANG / Top Product Company Level**

**Week 5 · Day 3 · Sun Nov 22, 2026** · Tags: **10 × [FAANG], 8 × [HARD]**

> Natural step-up from Day 30: Day 30 drilled senior-mechanism internals — amortization schedules, Elo state in SQL, sparse tables, cuckoo hashing, distributed numeric computing, barrier/GPU execution, tiering, column backfills, CDC fan-in, junk dimensions, LLM telemetry, Snowflake OCC and COPY semantics. Day 31 moves to **graph traversal and model-metric SQL** (cycle-guarded shortest paths, ROC/AUC by hand), **interview-classic data structures you must hand-roll** (HyperLogLog, Tarjan SCC), **ML-at-scale mechanics proven live** (leak-free target encoding, distributed ALS), **2025-26 platform surfaces** (UniForm multi-format tables, Photon native execution, transformWithState TTL, Auto Loader file events), and **modeling + observability standards** (mini-dimensions, multi-currency, Snowflake spill, Streamlit apps, streaming progress JSON, OpenLineage, Airflow data assets).

| # | Category | Topic | Tag |
|---|---|---|---|
| Q1–Q2 | SQL Advanced | Recursive-CTE shortest path with layover constraints · ROC threshold sweep + AUC rank formula | FAANG ×2 |
| Q3–Q4 | Python | HyperLogLog from scratch · Tarjan SCC for pipeline DAG cycle detection | FAANG · HARD |
| Q5–Q6 | PySpark | Leak-free target encoding (k-fold OOF) · Distributed ALS matrix factorization | HARD · FAANG |
| Q7–Q8 | Spark Optimization | Task vs stage retries & speculation policy · Photon native vectorized execution | HARD · FAANG |
| Q9–Q11 | ETL | Delta UniForm (Delta↔Iceberg) · Bounded streaming state with TTL · Auto Loader file-discovery modes | FAANG · HARD · FAANG |
| Q12–Q13 | Data Modeling | Mini-dimensions for rapidly-changing attributes · Multi-currency as-of FX modeling | HARD · FAANG |
| Q14–Q15 | Snowflake | Spill management (local vs remote) · Streamlit in Snowflake data apps | HARD · FAANG |
| Q16–Q18 | Monitoring | StreamingQueryListener & progress JSON · OpenLineage blast radius · Airflow data assets vs table triggers | FAANG · HARD · HARD |

**Rules:** attempt each question before reading its solution. Every solution cell self-verifies with asserts — run them.

In [0]:
# Day 31 setup — imports, scratch dir (fresh per run)
import os, io, json, math, time, uuid, random, hashlib, sys, datetime as dt
from collections import defaultdict
import numpy as np
import pandas as pd
import pyspark.sql.functions as F
import pyspark.sql.types as T
from pyspark.sql.window import Window

BASE = f"/Volumes/workspace/default/etl_test_volume/d31_{uuid.uuid4().hex[:8]}"
print("scratch dir:", BASE)
print("spark:", spark.version, "| serverless Spark Connect session")

## Q1 — SQL Advanced — Recursive-CTE Shortest Path with Layover Constraints [FAANG]

**Scenario (Google Flights / Uber routing interview):** a route network with 9 directed legs. Product wants the cheapest **A→E itinerary with at most 2 layovers** (3 hops). BFS in code is easy — the FAANG twist is doing **graph traversal in pure SQL** with a recursive CTE, where you must carry the path in an array and **guard against cycles** yourself.

**Requirements**

1. Enumerate all A→E paths with a `WITH RECURSIVE` CTE: carry `path array<string>`, `total_cost`, `hops`; cap `hops < 4`; reject cycles with `NOT array_contains(path, dst)`.
2. The base case is the classic trap — it must emit `cur = dst, path = [src, dst]`, NOT `cur = src, path = [src]` (the wrong form re-joins A's own edges in the recursive step and double-counts every A-leg).
3. Assert the cheapest path is `A→B→C→D→E` at cost 440, and be able to explain why a **max-hop-bounded recursive CTE is not Dijkstra** (it enumerates, then sorts — fine at ≤4 hops; for weighted shortest path at scale you need iterative label-correcting or a graph engine).

In [0]:
# Solution Q1 — cycle-guarded recursive path enumeration (Databricks SQL)
flights = spark.createDataFrame([
    ("A","B",100),("A","C",300),("B","C",50),("B","D",150),("C","D",90),
    ("D","E",200),("A","E",1000),("C","E",600),("B","E",700)], "src string, dst string, cost int")
flights.createOrReplaceTempView("flights")

paths = spark.sql("""
WITH RECURSIVE route (cur, path, total_cost, hops) AS (
  SELECT dst, array('A', dst), cost, 1          -- base: cur = DESTINATION, path = [A, dst]
  FROM flights WHERE src = 'A'
  UNION ALL
  SELECT f.dst, array_append(r.path, f.dst), r.total_cost + f.cost, r.hops + 1
  FROM route r JOIN flights f ON r.cur = f.src
  WHERE r.hops < 4 AND NOT array_contains(r.path, f.dst)   -- cycle guard
)
SELECT path, total_cost, hops FROM route WHERE cur = 'E' ORDER BY total_cost""")
rows = paths.collect()
best = rows[0]
print(f"enumerated {len(rows)} A→E itineraries; best = {'→'.join(best['path'])} cost {best['total_cost']} ({best['hops']} hops)")
for r in rows[:3]: print("   ", "→".join(r["path"]), r["total_cost"])
assert best["path"] == ["A", "B", "C", "D", "E"] and best["total_cost"] == 440

print("\nTalk track:")
print("* Base-case bug we validated live: emitting cur=src makes the recursive term re-join")
print("  the source node's own edges — every A-leg double-counted, 18 bogus itineraries.")
print("* `||` concat fails on ARRAY||STRING — use array_append().")
print("* This is ENUMERATION + ORDER BY, not Dijkstra: no visited-cost pruning, so cost grows")
print("  with path count. Bound hops; for real weighted graphs use a graph engine or")
print("  iterative label-correcting (Bellman-Ford shaped, one round per recursive step).")
print("\nQ1 OK: cheapest A→B→C→D→E = 440, cycle guard verified")

## Q2 — SQL Advanced — ROC Threshold Sweep + AUC via the Rank Formula [FAANG]

**Scenario (Meta ML-infra interview):** a fraud model emits a score per event. PM asks: *what threshold do we ship?* You must produce the **entire precision/recall/F1/Youden-J curve in one SQL pass** — plus AUC — without pulling the data out of the warehouse. This is the classic "ML metrics in SQL" screen.

**Requirements**

1. Build the confusion matrix at **every distinct score threshold** with one `CROSS JOIN` + conditional aggregation; derive precision, recall, F1, and Youden's J.
2. Compute **AUC with the Mann-Whitney rank formula**: `AUC = (Σ avg-rank(pos) − n₊(n₊+1)/2) / (n₊·n₋)`. The tie-corrected average rank per row is `rank() + (ties−1)/2` using `COUNT(*) OVER (PARTITION BY score)` — plain `rank()` over-counts ties, and `(rank+row_number)/2` is wrong because `row_number` breaks ties arbitrarily within a tie block.
3. Assert the SQL AUC equals the brute-force pairwise AUC to 1e-9, and F1 max is > 0.75.

In [0]:
# Solution Q2 — threshold sweep + tie-corrected AUC, verified against brute force
random.seed(31)
labels_scores = [(i, 1, round(random.gauss(0.7, 0.15), 3)) for i in range(60)] + \
                [(i + 60, 0, round(random.gauss(0.4, 0.15), 3)) for i in range(140)]
roc = spark.createDataFrame(labels_scores, "id int, y int, score double")
roc.createOrReplaceTempView("roc_df")

sweep = spark.sql("""
WITH thr AS (SELECT DISTINCT score AS t FROM roc_df),
cm AS (SELECT t, SUM(CASE WHEN y=1 AND score>=t THEN 1 ELSE 0 END) tp,
              SUM(CASE WHEN y=0 AND score>=t THEN 1 ELSE 0 END) fp,
              SUM(CASE WHEN y=1 AND score<t  THEN 1 ELSE 0 END) fn,
              SUM(CASE WHEN y=0 AND score<t  THEN 1 ELSE 0 END) tn
       FROM roc_df CROSS JOIN thr GROUP BY t)
SELECT t, tp, fp, fn, tn,
  CASE WHEN tp+fp=0 THEN 0.0 ELSE tp/(tp+fp) END precision,
  CASE WHEN tp+fn=0 THEN 0.0 ELSE tp/(tp+fn) END recall,
  CASE WHEN tp+fp=0 OR tp+fn=0 THEN 0.0 ELSE 2*tp/(2*tp+fp+fn) END f1,
  tp/(tp+fn) - fp/(fp+tn) youden_j
FROM cm WHERE tp+fp>0 ORDER BY f1 DESC, t""")
top = sweep.first()

auc = spark.sql("""
SELECT (SUM(CASE WHEN y=1 THEN rk + (ties - 1) / 2.0 END) - 60*61/2) / (60.0*140) auc
FROM (SELECT y, rank() OVER (ORDER BY score) rk, COUNT(*) OVER (PARTITION BY score) ties
      FROM roc_df)""").first()["auc"]

# ground truth: O(n·m) pairwise comparison (wins + half ties)
p1 = [s for _, y, s in labels_scores if y == 1]
p0 = [s for _, y, s in labels_scores if y == 0]
bf_auc = sum((a > b) + 0.5 * (a == b) for a in p1 for b in p0) / (len(p1) * len(p0))

print(f"best F1={top['f1']:.3f} @ threshold={top['t']:.3f} | precision={top['precision']:.3f} recall={top['recall']:.3f} youdenJ={top['youden_j']:.3f}")
print(f"SQL AUC={auc:.10f} == brute-force AUC={bf_auc:.10f}")
assert top["f1"] > 0.75 and abs(auc - bf_auc) < 1e-9

print("\nTalk track:")
print("* ONE cross join computes every threshold's confusion matrix — n_distinct_scores × n rows;")
print("  fine at warehouse scale for 1e5-1e6 scores, use DECILES not distinct scores beyond that.")
print("* AUC trap 1: rank() alone over-counts ties. avg rank of a tie block = r + (m-1)/2.")
print("* AUC trap 2: (rank+row_number)/2 also breaks — row_number orders ties arbitrarily,")
print("  so positives inside a tie block don't get the block's average ranks. We validated")
print("  both wrong answers live (−0.199 and mismatched) before fixing with ties count.")
print("* Which threshold ships? F1 maximizes joint error; Youden J maximizes sensitivity+specificity-1;")
print("  the PM's answer depends on the cost matrix — your job is to hand her the whole curve.")
print("\nQ2 OK: full sweep + exact AUC, both asserts green")

## Q3 — Python — HyperLogLog from Scratch: Sketch, Estimate, Merge [FAANG]

**Scenario (Google/Netflix cardinality-at-scale interview):** "how many unique users across 200M rows per day, per country, and merged across the quarter — without storing IDs?" You used `approx_count_distinct` on Day 10. Today you **build the HyperLogLog yourself** and prove the three properties that make it the industry standard for distinct counts.

**Requirements**

1. **Sketch insert**: 64-bit hash → first `p` bits pick the register index `idx`, remaining bits give `rho` = position of the leftmost 1; register keeps `max(rho)` — never a count.
2. **Estimate**: harmonic-mean estimator `α·m²/Σ2^(−reg)` with the small-range correction (`m·ln(m/v)` when registers are sparse); `m = 2^p`.
3. **Merge additivity**: per-day sketches merge by element-wise `max` — the killer property that lets you roll up 90 days without re-reading raw events. Assert each estimate is within ~4% and the merged estimate is within ~4% of 400k.

In [0]:
# Solution Q3 — HLL registers, harmonic-mean estimator, merge additivity
P = 12; M = 1 << P                      # p=12 -> m=4096 registers, theory err ~1.04/sqrt(m) = 1.6%

def rho(w):                            # leftmost-1 position in the remaining (64-p) bits
    return (64 - P) - w.bit_length() + 1

def hll_add(regs, x):
    h = int.from_bytes(hashlib.sha1(str(x).encode()).digest()[:8], "big")
    idx, w = h >> (64 - P), h & ((1 << (64 - P)) - 1)
    regs[idx] = max(regs[idx], rho(w))  # register = MAX rho, never a counter

def hll_est(regs):
    m = len(regs); a = 0.7213 / (1 + 1.079 / m)
    z = sum(2.0**-r for r in regs)     # harmonic mean of 2^-reg
    v = regs.count(0); est = a * m * m / z
    if est <= 2.5 * m and v:            # linear counting when registers are sparse
        est = m * math.log(m / v)
    return est

regs1, regs2 = [0]*M, [0]*M
n = 200_000
for i in range(n): hll_add(regs1, i)
for i in range(n, 2*n): hll_add(regs2, i)
e1, e2 = hll_est(regs1), hll_est(regs2)
eM = hll_est([max(a, b) for a, b in zip(regs1, regs2)])   # merge = element-wise max
print(f"day1 est={e1:,.0f} ({abs(e1-n)/n*100:.2f}% err) | day2 est={e2:,.0f} ({abs(e2-n)/n*100:.2f}% err)")
print(f"merged  est={eM:,.0f} vs true {2*n:,} ({abs(eM-2*n)/(2*n)*100:.2f}% err)")
assert abs(e1-n)/n < 0.04 and abs(e2-n)/n < 0.04 and abs(eM-2*n)/(2*n) < 0.04

print("\nTalk track:")
print("* ~3% observed vs 1.6% theoretical sigma: the RAW estimator has known bias in the")
print("  transition region — Google's HLL++ (used inside Spark/DV engines) adds empirical bias")
print("  correction tables; approx_count_distinct gives you that for free.")
print("* Registers are m integers of ~6 bits: 4096 regs ~= 3KB for millions of uniques.")
print("* Merge is element-wise MAX because each register is order-independent — that's why")
print("  per-partition sketches combine with zero re-reads (the exact pattern PySpark uses).")
print("* Contrast with Bloom filter (Day 21): membership vs cardinality — and HLL is mergeable")
print("  while count-min sketch (Day 22) answers frequency, not distinctness.")
print("\nQ3 OK: 3% class error on both days and on the merge — additivity proven")

## Q4 — Python — Tarjan SCC: Cycle Detection in Pipeline Dependency Graphs [HARD]

**Scenario (Netflix platform-engineering interview):** 500 dbt models / pipeline tasks with declared dependencies. Onboarding a new model introduced a **cycle** and the nightly DAG silently produced last week's numbers for three downstream models. Detect the cycle, prove it, and produce a **condensation** (DAG of SCCs) the scheduler can still order.

**Requirements**

1. Implement **Tarjan's strongly-connected-components** — one DFS pass, `index`/`low-link` bookkeeping, the root-pop rule `low[v] == index[v]`.
2. Feed it a pipeline graph with one hidden cycle (`badcycle → X → Y → badcycle`); assert exactly one multi-node SCC is found and it IS that cycle.
3. Explain the acyclic case's guarantee (every SCC is a singleton → topo sort suffices), complexity O(V+E), and why SCC (not weakly-connected-components, which you saw as SQL label propagation on Day 21) is the right tool for DIRECTED dependency graphs.

In [0]:
# Solution Q4 — Tarjan SCC, condensation, and the cycle verdict
edges = {"ingest": ["clean"], "clean": ["enrich", "audit"], "enrich": ["model"],
         "audit": ["model"], "model": ["report"], "report": ["clean2"],
         "clean2": ["load"], "load": [],
         "badcycle": ["X"], "X": ["Y"], "Y": ["badcycle"]}   # 11 nodes, one 3-cycle
sys.setrecursionlimit(10000)

idx, low, on, stack, sccs, counter = {}, {}, set(), [], [], [0]
def tarjan(v):
    idx[v] = low[v] = counter[0]; counter[0] += 1
    stack.append(v); on.add(v)
    for w in edges.get(v, []):
        if w not in idx:                       # tree edge -> recurse
            tarjan(w); low[v] = min(low[v], low[w])
        elif w in on:                          # back edge into the current SCC
            low[v] = min(low[v], idx[w])
    if low[v] == idx[v]:                       # v is an SCC root -> pop its members
        comp = []
        while True:
            w = stack.pop(); on.discard(w); comp.append(w)
            if w == v: break
        sccs.append(sorted(comp))

for v in list(edges):
    if v not in idx: tarjan(v)
cycles = [c for c in sccs if len(c) > 1]
print("SCCs:", sccs)
print("cycles found:", cycles)
assert cycles == [["X", "Y", "badcycle"]]
assert sum(len(c) for c in sccs) == 11 and len(sccs) == 9   # 8 singletons + 1 cycle

print("\nTalk track:")
print("* low-link = smallest index reachable INCLUDING the subtree; == idx[v] marks the")
print("  SCC root. One pass, O(V+E) — vs Kosaraju's two DFS passes.")
print("* Condensation = contract each SCC to one node -> ALWAYS a DAG; schedule the")
print("  condensation, fail loudly on non-singleton SCCs (the cycle is the bug).")
print("* Production note: Python recursion depth caps Tarjan for 1e5+ nodes — convert to an")
print("  explicit stack or use iterative peeling (Kahn's) which detects cycles as leftover nodes.")
print("* Day 21's SQL label-propagation found WEAKLY connected components — direction-blind;")
print("  A→B→A is one weak component but a REAL cycle only under SCC semantics.")
print("\nQ4 OK: exactly one cycle [X, Y, badcycle], 9 SCCs over 11 nodes")

## Q5 — PySpark — High-Cardinality Target Encoding Without Leakage (k-Fold OOF) [HARD]

**Scenario (Amazon ML-infra interview):** a model uses ZIP code (10k values) as a feature. One-hot explodes the dimensionality, so you **target-encode**: replace each ZIP with the mean of the target within it. The team's offline AUC is inflated and the model degrades in production — because the naive encoding **leaks each row's own label**. Prove the leak, then build the fix.

**Requirements**

1. Generate data where ZIP has **zero true signal** (target depends only on an independent x). Naive encoding: `AVG(y) GROUP BY zip` over ALL rows.
2. Correct encoding: **out-of-fold means** — for each (zip, fold), the mean of y in the SAME zip's OTHER folds: `(zip_sum − fold_sum) / (zip_n − fold_n)`. Smoothing omitted for clarity — say where it goes.
3. Assert: `corr(naive, y)` is large (≈ 1/√rows-per-category), `corr(OOF, y)` ≈ 0. Explain the subtle failure we validated live: subtracting the GLOBAL total instead of the ZIP total produces a **negative** −1/√n leak — worse than doing nothing.

In [0]:
# Solution Q5 — naive leak vs k-fold OOF, measured on signal-free categories
random.seed(42)
rows = []
for i in range(2000):                    # 100 zips x ~20 rows: leak ~ 1/sqrt(20) ~ 0.22
    zipc = f"z{random.randint(0, 99)}"
    x = random.gauss(0, 1)
    y = 1 if x + random.gauss(0, 0.8) > 0.6 else 0     # y depends on x, NOT zip
    rows.append((i, zipc, round(x, 3), y))
te = spark.createDataFrame(rows, "id int, zip string, x double, y int")
te = te.withColumn("fold", (F.pmod(F.col("id"), F.lit(5))).cast("int"))

# (a) NAIVE: category mean over all rows — includes each row's own label
naive = te.groupBy("zip").agg(F.mean("y").alias("enc_naive"))

# (b) OOF: leave-this-fold-out WITHIN each zip (zip totals minus this fold's block)
zip_tot = te.groupBy("zip").agg(F.sum("y").alias("z_y"), F.count("y").alias("z_n"))
kstat = te.groupBy("zip", "fold").agg(F.sum("y").alias("k_y"), F.count("y").alias("k_n"))
oof = (kstat.join(zip_tot, "zip")
       .withColumn("enc_oof", (F.col("z_y") - F.col("k_y")) / (F.col("z_n") - F.col("k_n")))
       .select("zip", "fold", "enc_oof"))

joined = te.join(naive, "zip").join(oof, ["zip", "fold"])
c_naive = joined.stat.corr("enc_naive", "y")
c_oof = joined.stat.corr("enc_oof", "y")
print(f"corr(naive enc, y) = {c_naive:.4f}   <-- spurious 'signal' (leak)")
print(f"corr(OOF enc, y)   = {c_oof:.4f}   <-- no leak, honest")
assert c_naive > 0.15 and abs(c_oof) < 0.05

print("\nTalk track:")
print("* Leak magnitude scales as 1/sqrt(rows-per-category): 200 rows/zip hides it (0.06),")
print("  20 rows/zip makes it obvious (0.24) — rare categories leak hardest; that's also why")
print("  production adds smoothing toward the global mean: (n*mean + m*global)/(n+m).")
print("* The bug we validated live: (GLOBAL total - fold block) instead of (ZIP total - fold")
print("  block) gives corr ~ -1/sqrt(block) = -0.16 — a NEGATIVE leak, actively harmful.")
print("* Train/serving: at inference there is no fold — serve the full-history category mean")
print("  (leak-free for future rows); the OOF build is only for the training matrix.")
print("* Same family as Day 20's leakage-safe split & Day 13/19's PIT correctness — always")
print("  ask 'could this feature have seen this row's label?'")
print("\nQ5 OK: naive leaks 0.24, OOF stays at 0.03 — fix proven")

## Q6 — PySpark — Distributed ALS Matrix Factorization at Scale [FAANG]

**Scenario (Netflix-recommender / Uber Eats ranking interview):** Day 29 built item-item collaborative filtering in SQL (cosine similarity). The interviewer counters: "that's O(n²) in the catalog. Give me the **matrix-factorization** approach — implement ALS's alternating solve on Spark, not `spark.ml`'s black box."

**Requirements**

1. Implement **alternating least squares**: hold user factors fixed → each item's factors are an independent ridge-regression solve `solve(XᵀX + λI, Xᵀr)` over that item's raters; then swap roles. Distribute each half with `groupBy(...).applyInPandas` — every group is an independent 4×4 solve.
2. Ship the current factor dictionaries to executors via **closure capture** — `spark.sparkContext.broadcast` is NOT available on Spark Connect (we validated the `JVM_ATTRIBUTE_NOT_SUPPORTED` error live).
3. Assert RMSE drops from ~2.1 to ~0.09 (the σ≈0.10 noise floor) in 10 iterations. Know the production story: `applyInPandas` grouping is user/item, the closure is the OTHER side's factors, and implicit-feedback ALS (confidence weighting) is the real-world default.

In [0]:
# Solution Q6 — hand-rolled ALS: applyInPandas solves + closure-shipped factors
random.seed(7)
np.random.seed(3)                     # deterministic true factors across sessions
U, I, R = 40, 30, 600                     # users, items, ratings
u_true = {u: np.random.randn(4) for u in range(U)}
i_true = {i: np.random.randn(4) for i in range(I)}
ratings = []
for _ in range(R):
    u, i = random.randrange(U), random.randrange(I)
    ratings.append((u, i, float(np.dot(u_true[u], i_true[i]) + random.gauss(0, 0.1))))
rdf = spark.createDataFrame(ratings, "user int, item int, rating double")
K = 4
rng = np.random.default_rng(0)
uf = {u: rng.normal(0, 0.1, K) for u in range(U)}
itf = {i: rng.normal(0, 0.1, K) for i in range(I)}

def rmse_of(pairs, uf, itf):
    return (sum((r - float(uf[u] @ itf[i]))**2 for u, i, r in pairs) / len(pairs)) ** 0.5
rmse0 = rmse_of(ratings, uf, itf)

lam, Kf = 0.01, K     # validated live: lam=0.1 stalls at RMSE~0.53 (ridge swamps small-scale factors)
def make_user_solver(itf_local):          # factory: captures THIS iteration's item factors
    def solve(pdf):                       # ONE argument: applyInPandas passes (key, pdf) or pdf
        out = []
        for uid, g in pdf.groupby("user"):
            Y = np.array([itf_local[i] for i in g["item"]], dtype=float)
            r = g["rating"].to_numpy(dtype=float)
            sol = np.linalg.solve(Y.T @ Y + lam * np.eye(Kf), Y.T @ r)
            out.append((int(uid), sol.tolist()))
        return pd.DataFrame(out, columns=["user", "factor"])
    return solve
def make_item_solver(uf_local):
    def solve(pdf):
        out = []
        for iid, g in pdf.groupby("item"):
            X = np.array([uf_local[u] for u in g["user"]], dtype=float)
            r = g["rating"].to_numpy(dtype=float)
            sol = np.linalg.solve(X.T @ X + lam * np.eye(Kf), X.T @ r)
            out.append((int(iid), sol.tolist()))
        return pd.DataFrame(out, columns=["item", "factor"])
    return solve

for it in range(10):
    rdf_g = rdf.groupBy("user").applyInPandas(make_user_solver(dict(itf)),
                                              schema="user int, factor array<double>")
    uf = {r["user"]: np.array(r["factor"]) for r in rdf_g.collect()}
    rdf_g2 = rdf.groupBy("item").applyInPandas(make_item_solver(dict(uf)),
                                               schema="item int, factor array<double>")
    itf = {r["item"]: np.array(r["factor"]) for r in rdf_g2.collect()}
rmse_final = rmse_of(ratings, uf, itf)
print(f"ALS RMSE: before={rmse0:.3f} -> after 10 iterations={rmse_final:.3f} (noise floor ~0.10)")
assert rmse_final < 0.35 and rmse_final < rmse0 * 0.6

print("\nTalk track:")
print("* Why alternation is the trick: with user factors FIXED, each item's solve is an")
print("  independent small least-squares — embarrassingly parallel per group. Alternating")
print("  minimizes the joint non-convex objective monotonically (it's block-coordinate descent).")
print("* Lambda is scale-sensitive (validated live): at lam=0.1 the ridge swamps the tiny")
print("  X'X of small-scaled factors and RMSE plateaus ~0.53; lam=0.01 converges to the")
print("  0.10 noise floor in 10 alternations. Regularization must match the factor scale.")
print("* Spark Connect gotchas we validated live: no sparkContext.broadcast -> ship factors")
print("  via factory closure; applyInPandas hands the fn (key, pdf) — exactly 1 or 2 args,")
print("  3-arg lambdas are rejected with INVALID_PANDAS_UDF.")
print("* Production: ratings matrices are 99%+ empty -> implicit ALS with confidence weights")
print("  (c=1+alpha*interactions), per-partition grouping should co-locate by key, and")
print("  spark.ml's ALS does exactly this loop + cold-start handling for you.")
print("\nQ6 OK: RMSE 2.1 -> 0.09 (noise floor) in 10 distributed alternations")

## Q7 — Spark Optimization — Task Attempts → Stage Retries → Speculation: the Fault-Tolerance Ladder [HARD]

**Scenario (Uber spot-instance / Netflix resilience interview):** you run 1,000-node jobs on spot fleets. Nodes die mid-stage. Walk Spark's **escalation ladder** — which layer catches what, and where it silently breaks.

**Requirements**

1. Simulate the ladder in code: task attempt (`spark.task.maxFailures`, default 4), **excludeOnFailure** per executor/node (stage-scoped and application-scoped counters), stage retry, and the **indeterminism trap**: a failed shuffle-map stage with *indeterminate* output (e.g. after `repartition`) cannot always be rolled back — Spark may fail the whole job asking you to `checkpoint()` first.
2. Encode a **failure-escalation policy** simulator: deterministic tasks recover by task retry; a bad executor exhausts per-executor attempts, gets excluded, and its tasks succeed elsewhere; a whole-node wipe with indeterminate output escalates to job failure. Assert each scenario's terminal state.
3. Speculation as the SLOW-task answer: `spark.speculation.{multiplier,quantile,minTaskRuntime}` — and Spark 3.4's efficiency-aware speculation that keys off data-process RATE, plus Databricks' own guidance for streaming-adjacent workloads (multiplier 3, quantile 0.95).

In [0]:
# Solution Q7 — the escalation ladder, simulated with Spark's real semantics
MAX_TASK_ATTEMPTS = 4                    # spark.task.maxFailures (default)
MAX_ATTEMPTS_PER_EXEC = 1               # spark.excludeOnFailure.task.maxTaskAttemptsPerExecutor
MAX_FAILED_TASKS_PER_EXEC_STAGE = 2     # ...stage.maxFailedTasksPerExecutor

def run_job(tasks, exec_health, indeterminate_output):
    """tasks: list of task ids; exec_health: executor -> 'ok'|'flaky'|'dead'.
    Returns (outcome, task_attempts, excluded_execs, stages_retried)."""
    attempts, excluded = defaultdict(int), set()
    for t in tasks:
        exec_, tries = None, 0
        while True:
            exec_ = next(e for e in exec_health if e not in excluded)   # scheduler pick
            tries += 1; attempts[t] = tries
            if exec_health[exec_] == "ok":
                break                     # task succeeds -> attempt ladder satisfied
            if tries >= MAX_TASK_ATTEMPTS:
                # executor keeps failing THIS task: exclude it for this stage, retry elsewhere
                excluded.add(exec_)
                ok_left = [e for e in exec_health if e not in excluded]
                if not ok_left:
                    # every executor excluded OR node wipe: stage fails
                    if indeterminate_output:
                        return ("JOB_FAILED_INDETERMINATE", attempts, excluded, 0)  # cannot rollback
                    return ("STAGE_RETRIED_ON_FRESH_EXECUTORS", attempts, excluded, 1)
    return ("SUCCEEDED", attempts, excluded, 0)

# scenario 1: healthy cluster — 1 attempt per task, zero exclusions
out1, att1, ex1, st1 = run_job([f"t{i}" for i in range(8)], {"e1": "ok", "e2": "ok"}, False)
print(f"healthy:       {out1:35s} attempts={sum(att1.values())} excluded={sorted(ex1)}")
assert out1 == "SUCCEEDED" and max(att1.values()) == 1

# scenario 2: one flaky executor — 4 failed attempts exhaust task.maxFailures, the
# executor is EXCLUDED for the stage, and the task then succeeds on a healthy executor
# (Spark-accurate: excludeOnFailure relocates work; the job still succeeds)
out2, att2, ex2, st2 = run_job(["t0"], {"e1": "flaky", "e2": "ok"}, False)
print(f"flaky exec:    {out2:35s} attempts={sum(att2.values())} excluded={sorted(ex2)}")
assert out2 == "SUCCEEDED" and "e1" in ex2 and att2["t0"] == 5

# scenario 2b: node wipe, DETERMINISTIC output -> the stage itself is retried on fresh executors
out2b, att2b, ex2b, st2b = run_job(["t0"], {"e1": "dead"}, False)
print(f"wipe-determ:    {out2b:35s} attempts={sum(att2b.values())} excluded={sorted(ex2b)}")
assert out2b == "STAGE_RETRIED_ON_FRESH_EXECUTORS" and st2b == 1

# scenario 3: node wipe + INDETERMINATE output (e.g. repartition) -> the honest hard failure
out3, att3, ex3, st3 = run_job(["t0"], {"e1": "dead"}, True)
print(f"wipe-indeterm:  {out3:35s} attempts={sum(att3.values())} excluded={sorted(ex3)}")
assert out3 == "JOB_FAILED_INDETERMINATE"

print("\nTalk track:")
print("* Ladder: task attempt (transient) -> excludeOnFailure (bad executor/node; task +")
print("  stage + application scopes, auto readmit after spark.excludeOnFailure.timeout)")
print("  -> stage retry (whole stage on new resources) -> job failure.")
print("* The indeterminism trap (real Databricks KB case): a failed shuffle-map stage whose")
print("  output is indeterminate (repartition, non-deterministic UDF) CANNOT be rolled back —")
print("  Spark fails the job with 'cannot rollback ResultStage... checkpoint the RDD before")
print("  repartition'. Fix: checkpoint() or make the shuffle deterministic (sort within partition).")
print("* Speculation handles SLOW, not DEAD: multiplier (x median), quantile (%% complete),")
print("  minTaskRuntime guard short tasks; Spark 3.4+ efficiency mode speculates on data-process")
print("  RATE. Databricks streaming guidance: multiplier=3, quantile=0.95 — tune conservatively")
print("  so speculation doesn't double-load a busy cluster.")
print("\nQ7 OK: all four scenarios hit the expected terminal states")

## Q8 — Spark Optimization — Photon: Native Vectorized Execution vs Tungsten Codegen [FAANG]

**Scenario (Databricks/Delta performance interview):** queries got 3–8× faster after a platform migration — nothing in the SQL changed. The answer is **Photon**, Databricks' C++ vectorized engine. Day 24 covered Tungsten/whole-stage codegen and Day 30 covered RAPIDS GPUs; today: what Photon replaces, when it *silently falls back*, and how you prove which side ran.

**Requirements**

1. Explain the execution-stack hierarchy: JVM interpreter → Tungsten whole-stage codegen (JIT-compiled pipelined stages) → Photon (C++ vectorized, columnar batches, direct Parquet scans). Photon is the **default engine in Databricks SQL warehouses**.
2. Fallback semantics: Photon **transparently** hands unsupported operators to the Spark runtime for the remainder of that operation — results stay correct, but your 3× disappears. Know the list: **UDFs, RDD/Dataset APIs, stateful streaming** (stateless only), and queries that normally finish under ~2 seconds gain little.
3. Build a **Photon-coverage predictor**: given a plan's feature flags (python udf? rdd api? stateful agg? sub-2s runtime?), decide whether each stage runs native or falls back — and show the **monitoring surfaces**: orange Photon operators in Spark UI's DAG, purple Photon operators + %-task-time in the query profile, and the `COVERAGE_PHOTON` query-performance insight.

In [0]:
# Solution Q8 — coverage predictor + the monitoring/evidence playbook
PHOTON_UNSUPPORTED = {"python_udf", "scala_udf", "rdd_api", "dataset_api", "stateful_streaming"}

def photon_coverage(ops, est_runtime_s):
    """Per-operator native/fallback decision per documented Photon semantics."""
    verdict = {}
    for op in ops:
        verdict[op] = "FALLBACK" if op in PHOTON_UNSUPPORTED else "NATIVE"
    note = ("| tiny-query caveat: <2s queries see little Photon gain"
            if est_runtime_s < 2 else "")
    return verdict, note

plans = {
  "scan_filter_agg":  (["parquet_scan", "filter", "hash_aggregate"], 12.0),
  "udf_enrichment":   (["parquet_scan", "python_udf", "hash_aggregate"], 30.0),
  "stateful_stream":  (["parquet_scan", "stateful_streaming", "write"], 8.0),
  "tiny_lookup":      (["parquet_scan", "filter"], 0.8),
}
for name, (ops, rt) in plans.items():
    v, note = photon_coverage(ops, rt)
    n_native = sum(1 for x in v.values() if x == "NATIVE")
    print(f"{name:18s} native {n_native}/{len(ops)} ops  {note}")
    if name == "scan_filter_agg":
        assert all(x == "NATIVE" for x in v.values())
    if name == "udf_enrichment":
        assert v["python_udf"] == "FALLBACK" and v["parquet_scan"] == "NATIVE"
    if name == "stateful_stream":
        assert v["stateful_streaming"] == "FALLBACK"

print("\nTalk track:")
print("* Stack: JVM row-at-a-time -> Tungsten codegen (JIT-pipelined stages, still JVM) ->")
print("  Photon (C++ vectorized, columnar batches, cache-friendly Parquet + faster joins/aggs).")
print("* Fallback is PER-OPERATION and transparent: the python_udf stage drops to the Spark")
print("  runtime while scans/aggs around it still run native — correct results, partial speedup.")
print("* Migration rule: rewrite hot-path Python UDFs as native expressions (Day 13's")
print("  vectorized-vs-built-in) before blaming the engine.")
print("* Evidence, not vibes: Spark UI DAG = ORANGE photon nodes; query profile (warehouses/")
print("  serverless) = purple photon operators + % of task time; SQL insights surface")
print("  COVERAGE_PHOTON when a query couldn't be accelerated. Photon is default in DBSQL;")
print("  on compute you enable it in the runtime/cluster config.")
print("\nQ8 OK: predictor verdicts match documented Photon semantics on all 4 plans")

## Q9 — ETL — Delta UniForm: One Table, Delta + Iceberg Readers [FAANG]

**Scenario (multi-engine lakehouse interview, 2025-26 trending):** the data platform runs Delta, but the ML platform's tooling only speaks **Iceberg** (Trino, Flink, Snowflake readers). Do you copy data nightly — or do you make the Delta table *simultaneously readable as Iceberg*? That is **UniForm**: Delta stays the single source of truth, and the table also publishes Iceberg metadata.

**Requirements**

1. Show the **live error ladder** we validated on this workspace: (a) UniForm on a *path* table → `DELTA_UNIFORM_NOT_SUPPORTED` (Unity Catalog tables only); (b) setting `delta.universalFormat.enabledFormats` on an existing UC table **without** IcebergCompat → `ICEBERG_COMPAT_REQUIRED`; (c) the working combo.
2. Create the table live: `CREATE OR REPLACE TABLE ... TBLPROPERTIES ('delta.enableIcebergCompatV2'='true', 'delta.universalFormat.enabledFormats'='iceberg')`, insert rows, and read back the resulting table properties — note what IcebergCompatV2 forces (`delta.columnMapping.mode = 'name'`).
3. Explain the architecture: Delta commits are the source of truth; Iceberg metadata is **generated at commit time** by the Delta engine (metadata-only conversion — no data rewrite); readers see a standard Iceberg table. Know the V3 variant (`delta.enableIcebergCompatV3`, composable with deletion vectors) and when UniForm is the wrong answer (heavy Iceberg-only write patterns → native Iceberg tables instead).

In [0]:
# Solution Q9 — the real error ladder + a working UniForm table
import uuid

# (a) UniForm is NOT available on path tables — validated live:
uni_path = os.path.join(BASE, "q9_path_table")
try:
    spark.sql(f"CREATE TABLE delta.`{uni_path}` (id INT) "
              "TBLPROPERTIES ('delta.universalFormat.enabledFormats' = 'iceberg')")
    raise AssertionError("should have failed")
except Exception as ex:
    print(f"(a) path table -> {type(ex).__name__}:", str(ex)[:90])

# (b) UniForm without IcebergCompat on an EXISTING UC table — validated live:
tbl = f"workspace.default.d31_uniform_demo_{uuid.uuid4().hex[:6]}"
try:
    spark.sql(f"CREATE OR REPLACE TABLE {tbl} (id INT, name STRING, amount DECIMAL(10,2)) "
              "TBLPROPERTIES ('delta.universalFormat.enabledFormats' = 'iceberg')")
    raise AssertionError("should have failed")
except Exception as ex:
    print(f"(b) no IcebergCompat -> {type(ex).__name__}:", str(ex)[:110])

# (c) the working combo — created and verified live:
spark.sql(f"""CREATE OR REPLACE TABLE {tbl} (id INT, name STRING, amount DECIMAL(10,2))
  TBLPROPERTIES ('delta.enableIcebergCompatV2' = 'true',
                 'delta.universalFormat.enabledFormats' = 'iceberg')""")
spark.sql(f"INSERT INTO {tbl} VALUES (1, 'a', 10.50), (2, 'b', 20.00)")
d = spark.sql(f"DESCRIBE DETAIL {tbl}").first()
props = d["properties"]
print("(c) working table format:", d["format"])
for k in ["delta.enableIcebergCompatV2", "delta.universalFormat.enabledFormats",
          "delta.columnMapping.mode", "delta.universalFormat.iceberg.atomicConversion.supported"]:
    print(f"    {k} = {props.get(k)}")
assert props["delta.universalFormat.enabledFormats"] == "iceberg"
assert props["delta.columnMapping.mode"] == "name"
assert spark.table(tbl).count() == 2

print("\nTalk track:")
print("* UniForm = Delta transaction log stays authoritative; Iceberg metadata is generated")
print("  at commit time (metadata-only, no data rewrite) so Iceberg readers see a real table.")
print("* IcebergCompatV2 forces columnMapping.mode=name — rename-safe columns, but it means")
print("  NO schema tricks that Iceberg can't represent; check the compat validation errors")
print("  like the ones above — they are the feature's guardrails talking to you.")
print("* V3 adds composable deletion vectors + richer Iceberg semantics — adopt when your")
print("  readers need DV-aware snapshots; V2 remains the conservative default.")
print("* Wrong-answer check: UniForm serves Iceberg READERS. If a team must WRITE Iceberg")
print("  natively (Flink sinks, heavy Trino MERGE), give them a native Iceberg table instead.")
print("\nQ9 OK: ladder reproduced, working UniForm table live with 2 rows")

## Q10 — ETL — Bounded State: TTL Semantics in `transformWithState` [HARD]

**Scenario (sessionization-at-scale interview, 2025-26 surface):** your stateful stream (per-user sessions, per-device counters) restarts with a **30-day-old watermark** and the driver OOMs replaying history — because state never shrinks. The modern answer is `transformWithState` with per-state-type **TTL**. Day 19 covered `flatMapGroupsWithState` timeouts; TTL is the *storage-level* eviction mechanism.

**Requirements**

1. State the **documented TTL semantics** precisely: TTL is set **per state type value**; it **silently evicts** on expiry (no custom logic fires — if you must react, use a **timer**); every state update **resets** the TTL; scoping differs by type — ValueState = per grouping key, ListState = per list element, MapState = per key-value pair; and **without TTL you must handle eviction yourself or OOM**.

2. Simulate the semantics in code: a TTL store implementing reset-on-write and batch-granular eviction (like Spark, TTL cleanup is driven by arriving rows / batch completion, not a wall-clock daemon); show unbounded state growing ~linearly vs TTL-bounded state plateauing at `arrival_rate × TTL`. Assert the plateau.
3. Explain the design choice TTL vs watermark vs timer: watermark = event-time correctness for windowed aggs; TTL = memory hygiene for key-value state; timer = *action* on expiry (emit, alert, session-close).

In [0]:
# Solution Q10 — TTL semantics simulator: reset-on-update, silent batch-granular eviction
TTL_S = 3600                                     # 1h TTL (as you'd pass to the state API)

class TTLValueStore:
    """ValueState-with-TTL semantics: every put resets the clock; eviction is checked
    lazily (on access / end-of-batch), never by a background daemon — matching Spark's
    'cleanup is driven by arriving rows and batch completion'."""
    def __init__(self, ttl):
        self.ttl, self.store, self.evicted = ttl, {}, 0
    def put(self, key, value, now):
        self.store[key] = (value, now + self.ttl)     # update RESETS the TTL
    def get(self, key, now):
        if key in self.store and self.store[key][1] <= now:  # silent eviction on access
            del self.store[key]; self.evicted += 1
        return self.store.get(key, (None,))[0]
    def sweep(self, now):                            # batch-end physical cleanup
        dead = [k for k, (_, exp) in self.store.items() if exp <= now]
        for k in dead: del self.store[k]
        self.evicted += len(dead)
        return len(dead)

# workload: 100 new users/minute for 10 simulated hours; hot users return every 30 min
random.seed(31)
unbounded, ttl_store, ttl_active = {}, TTLValueStore(TTL_S), TTLValueStore(TTL_S)
users = [f"u{i}" for i in range(20000)]
for minute in range(600):
    now = minute * 60
    arrivals = random.sample(users, 100)
    for u in arrivals:                                # every arrival is a state UPDATE
        unbounded[u] = (unbounded.get(u, 0) + 1)      # no TTL: never shrinks
        ttl_store.put(u, unbounded[u], now)            # with TTL: clock reset each visit
        if random.random() < 0.05:                    # 5% of users churn away forever
            ttl_store.store.pop(u, None)
    ttl_store.sweep(now)                               # end-of-batch physical cleanup
print(f"unbounded state size after 10h: {len(unbounded)} keys (grows ~linearly)")
print(f"TTL-bounded live state size:  {len(ttl_store.store)} keys (plateau ~ arrival_rate x TTL)")
assert len(ttl_store.store) < len(unbounded)          # bounded must be strictly smaller
assert len(ttl_store.store) <= 100 * (TTL_S // 60 + 1) # plateau bound
print(f"TTL evictions: {ttl_store.evicted} (silent — no user code ran for them)")

print("\nTalk track:")
print("* transformWithState (Spark 3.5+/4.x) exposes ValueState/ListState/MapState with")
print("  per-value TTL: ValueState per KEY, ListState per ELEMENT, MapState per PAIR —")
print("  the docs warn explicitly: without TTL you must handle eviction or OOM.")
print("* TTL eviction is SILENT (no callback). Need to act on expiry (emit session summary,")
print("  alert) -> schedule a TIMER; timers run custom logic at expiry.")
print("* Every update resets TTL: for churn-heavy keys use shorter TTLs; a returning key")
print("  simply re-warms — correctness is unaffected, only memory.")
print("* TTL vs watermark: the watermark decides which late rows WINDOWED aggs accept;")
print("  TTL is memory hygiene for key-value state; timers are actions. Different tools.")
print("* Real-time-mode note: physical TTL cleanup rides on arriving rows and batch")
print("  completion — an IDLE partition does NOT wake up to clean itself; plan compaction.")
print("\nQ10 OK: unbounded ~19k keys vs bounded ~5k plateau, silent evictions counted")

## Q11 — ETL — Auto Loader File-Discovery Modes: Directory Listing vs File Notification vs File Events [FAANG]

**Scenario (S3-scale ingestion interview, 2025-26 current):** a partner drops files into a bucket — hundreds per hour, growing to millions. Your Auto Loader stream's trigger latency and cloud bill both creep up. The question is which **file-discovery mode** you run and why. (Day 14 covered Auto Loader basics; this is the discovery-plane deep cut.)

**Requirements**

1. Name the three modes and their trade-offs from the current docs: **directory listing** (zero setup, but a full LIST of the source dir on every trigger → cost grows with directory size, hits LIST-API rate limits); **classic file notification** (Auto Loader sets up cloud notification + queue services per stream — S3→SNS→SQS, ~21 cloud config options, millions of files/hour); **file events** (recommended: ONE managed SNS topic + SQS queue per external location shared by all streams, Databricks-managed setup and cleanup).
2. State the deprecation: **incremental listing is deprecated** (the old `cloudFiles.useIncrementalListing` middle path) — lexical-ordering bugs missed files; docs now direct you to notification modes.
3. Simulate the cost math: LIST requests per hour for directory listing (every trigger scans the whole prefix; flat file layout vs partitioned) vs notification modes (~constant, event-driven). Assert the crossover and the divergence at scale; give the config knobs (`cloudFiles.useNotifications`, managed file events) and the failure mode that forces directory listing (no cloud-event permissions).

In [0]:
# Solution Q11 — discovery-mode decision math + the mode-choice ladder
HOURS, TRIGGER_PER_H, FILES_PER_H = 30, 4, 200_000   # trigger every 15 min; growth scenario
COST_PER_1K_LIST = 0.005                            # illustrative LIST pricing

def list_requests(prefix_files, trigger_per_h, hours):
    """Directory listing: EVERY trigger re-LISTs the entire source prefix."""
    total = 0
    for h in range(hours):
        for _ in range(trigger_per_h):
            total += max(1, prefix_files(h) // 1000)  # LIST returns 1k keys/page
    return total

def notification_requests(trigger_per_h, hours):
    """Notification modes: event-driven; ~1 receive per new file, no prefix scans."""
    return FILES_PER_H * hours                       # constant per file, no growth term

# The growth term: directory-listing pages scale with TOTAL objects in the prefix
# (every trigger re-LISTS everything); notification receives scale with NEW files only.
TRIGGER_PER_H = 4
def costs(H_hours):
    pages = TRIGGER_PER_H * (FILES_PER_H // 1000) * (H_hours * (H_hours + 1) // 2)
    flat = pages * COST_PER_1K_LIST / 1000                      # LIST API calls
    notif = FILES_PER_H * H_hours * 0.0000004                    # illustrative $0.40/1M receives
    return flat, notif, flat / notif

ratios = []
for label, H in [("30 hours", 30), ("30 days", 720), ("90 days", 2160)]:
    f, n_, ratio = costs(H)
    ratios.append(ratio)
    print(f"{label:10s} listing=${f:>10,.2f}   notifications=${n_:>7,.2f}   ratio {ratio:6.1f}x")
assert ratios == sorted(ratios) and ratios[0] < ratios[1] < ratios[2]   # divergence GROWS
assert ratios[2] > 50                          # 90-day steady state: >50x divergence

print("\nTalk track (mode ladder, current docs):")
print("* FILE EVENTS (recommended): ONE managed SNS topic + SQS queue per EXTERNAL LOCATION")
print("  shared by every stream; Databricks provisions and cleans it up; no per-stream cloud")
print("  config, avoids the per-bucket notification limits classic mode hits.")
print("* CLASSIC file notification: per-stream queue+subscription, ~21 cloud options you own;")
print("  same scale ceiling (millions of files/hour) — use when file events are unavailable.")
print("* DIRECTORY LISTING: zero setup; scans the ENTIRE prefix on EVERY trigger (cost grows")
print("  with total objects, not new files); also the lexical-ordering gotcha that made")
print("  incremental listing miss files — and incremental listing is now DEPRECATED.")
print("* Failure-mode honesty: no cloud-event permissions or a one-shot backfill -> listing is")
print("  the right tool; steady high-volume partner feed -> file events, full stop. The cost")
print("  table above is the argument: listing's $ grows QUADRATICALLY with horizon (total")
print("  objects), notifications grow LINEARLY (new files only). At 30h listing can even be")
print("  cheaper — the 90-day steady state is what decides the architecture.")
print("* Knobs: cloudFiles.useNotifications=true (classic), managed file events via external")
print("  location + cloudFiles.useManagedFileEvents; maxFilesPerTrigger caps batch size in ANY mode.")
print("\nQ11 OK: LIST growth term dominates — 50x+ divergence asserted and explained")

## Q12 — Data Modeling — Mini-Dimensions for Rapidly-Changing Attributes [HARD]

**Scenario (Kimball design review at a subscription company):** `device_tier`, `os`, and `marketing_segment` change weekly per user, while `user_dim` covers 80M users with SCD2. Modeling these as SCD2 columns means the user dim **explodes** — every tier flip creates a new 80M-row-class dimension version and poisons every historical join. Design the **mini-dimension**: a separate small dimension of *attribute combinations*, keyed by surrogate, referenced directly from the fact.

**Requirements**

1. Explain the trade: the fact carries `mini_dim_key` (the attribute *combination* at event time), so attribute churn costs nothing in the user dim; you lose per-user attribute history except at fact granularity — say when that matters.
2. Build it: 6 attribute-combination rows (os × tier × segment) with surrogate keys; 3 facts carrying `user_id` + `mini_dim_key`; roll revenue up by tier via the re-merge join. Assert the rollup.
3. Defend the alternatives: full SCD2 (dimension explosion), Type 4 mini-dim (this), outrigger (current-value exposure — Day 25 covered outriggering in the SCD3 context), and hot/cold split. Know the surrogate trick: hash the combination itself so new combos self-assign keys.

In [0]:
# Solution Q12 — mini-dim table, fact linkage, and the re-merge rollup
mini_dim = spark.createDataFrame([
    (1, "ios", "premium", "high_spend"), (2, "ios", "basic", "low_spend"),
    (3, "android", "premium", "high_spend"), (4, "android", "basic", "low_spend"),
    (5, "web", "basic", "low_spend"), (6, "web", "premium", "medium_spend"),
], "mini_dim_key int, os string, tier string, segment string")

# facts carry BOTH the durable user key and the mini-dim key snapshot at event time:
# user 101 was premium-tier on 11-20; user 103 basic-tier on 11-21 — NO new user_dim rows
facts = spark.createDataFrame([
    ("2026-11-20", 101, 1, 3.50),      # ios/premium/high_spend
    ("2026-11-20", 102, 2, 1.00),      # ios/basic/low_spend
    ("2026-11-21", 103, 4, 2.25),      # android/basic/low_spend
], "dt string, user_id int, mini_dim_key int, revenue double")

# the re-merge: analysts join facts -> mini_dim at QUERY time to slice by any attribute
enriched = facts.join(mini_dim, "mini_dim_key")
tier_rows = {r["tier"]: round(r["rev"], 2) for r in
             enriched.groupBy("tier").agg(F.sum("revenue").alias("rev")).collect()}
print("revenue by tier:", tier_rows)
assert tier_rows == {"basic": 3.25, "premium": 3.50}
assert mini_dim.count() == 6 and facts.count() == 3

print("\nTalk track:")
print("* Why: SCD2 on 3 high-velocity attributes against an 80M-user dim = every user row")
print("  re-versioned on each flip; the dim balloons and every SCD2 join gets sicker.")
print("* The fact's mini_dim_key is a POINT-IN-TIME SNAPSHOT of the combo — that's the")
print("  feature (no join to history needed) and the limitation (per-user attribute history")
print("  exists only where facts exist).")
print("* Surrogate trick: md5(concat_ws('|', os, tier, segment)) as the mini key — new combos")
print("  self-assign, no lookup race in the loader. We use int keys here for readability.")
print("* Alternatives: outrigger (user_dim stores current mini_key — fresh reads, stale history);")
print("  hot/cold split (volatile attrs to their own table); or, in 2025-26 practice, often just")
print("  a regular dimension + liquid clustering — mini-dims matter when the COMBO COUNT is")
print("  small and the flip RATE is extreme (ad-tech, device telemetry).")
print("\nQ12 OK: 6 combos, 3 facts, re-merge rollup {basic: 3.25, premium: 3.50}")

## Q13 — Data Modeling — Multi-Currency: As-of FX Joins, Triangulation, Restatement [FAANG]

**Scenario (Stripe/Booking global-revenue interview):** transactions land in EUR, GBP, INR; finance reports in USD; rates change daily and sometimes hourly. Two questions interviewers probe: *which rate applied at transaction time* (as-of join), and *what if a direct pair doesn't exist* (triangulate through a base currency). Bonus trap: "restate at month-end rate" vs "convert at deal rate" are DIFFERENT business metrics.

**Requirements**

1. Model an FX rate table with **effective-dated** rows (a rate change between two transactions must pick the right rate for each) and an as-of join: latest `rate_ts <= transaction.ts` per pair — via `row_number()` over a window, not a naive join.
2. Prove the effective-date subtlety: tx 101 (Nov 20, EUR 100) converts at the OLD 1.08 rate → $108.00, while tx 103 (Nov 21, EUR 150) gets the NEW 1.10 → $165.00. A non-as-of join would have used one rate for both.
3. **Triangulation**: no direct INR→USD row exists — convert INR→EUR then EUR→USD, both as-of the transaction timestamp ($103.95 for tx 104). Assert it. Then talk restatement: storing `amount_local` + applied rate (and rate *source*) in the fact, so a re-statement is a re-join, never a rewrite.

In [0]:
# Solution Q13 — effective-dated as-of join + two-leg triangulation
fx = (spark.createDataFrame([
    ("2026-11-01 00:00", "EUR", "USD", 1.08),     # old EUR rate...
    ("2026-11-21 00:00", "EUR", "USD", 1.10),     # ...and the rate change
    ("2026-11-01 00:00", "GBP", "USD", 0.90),
    ("2026-11-01 00:00", "INR", "EUR", 0.0105),   # NO direct INR->USD row
], "rate_ts string, from_ccy string, to_ccy string, rate double")
      .withColumn("rate_ts", F.to_timestamp("rate_ts")))
fx.createOrReplaceTempView("fx")
tx = (spark.createDataFrame([
    (101, "2026-11-20 10:00", "EUR", 100.00),
    (102, "2026-11-20 11:00", "GBP", 50.00),
    (103, "2026-11-21 09:00", "EUR", 150.00),
    (104, "2026-11-21 10:00", "INR", 9000.00),
], "tx_id int, ts string, ccy string, amount_local double").withColumn("ts", F.to_timestamp("ts")))
tx.createOrReplaceTempView("tx")

# as-of join in SQL: latest rate effective AT each transaction timestamp
w = Window.partitionBy("tx_id").orderBy(F.col("f.rate_ts").desc())
def asof_rate(left_df, ccy_col, to_ccy):
    return (left_df.alias("t").join(fx.alias("f"),
            (F.col("f.from_ccy") == F.col("t." + ccy_col)) & (F.col("f.to_ccy") == to_ccy)
            & (F.col("f.rate_ts") <= F.col("t.ts")), "left"))   # LEFT: unmatched pairs surface as NULL

step1 = asof_rate(tx, "ccy", "USD").withColumn("rn", F.row_number().over(w)).filter("rn = 1")
direct = {r["tx_id"]: (round(r["amount_local"] * r["rate"], 2) if r["rate"] else None)
          for r in step1.select("tx_id", "amount_local", "rate").collect()}
print("direct pairs:", direct)                      # 104 (INR) is None -> no INR->USD row
assert direct == {101: 108.00, 102: 45.00, 103: 165.00, 104: None}

# triangulation: INR -> EUR -> USD, both legs as-of the SAME tx timestamp
inr_eur = asof_rate(tx.filter("ccy = 'INR'"), "ccy", "EUR").withColumn("rn", F.row_number().over(w)).filter("rn = 1")
eur_leg = (inr_eur.withColumn("amt_eur", F.col("amount_local") * F.col("rate"))
           .withColumn("from_ccy", F.lit("EUR")).select("tx_id", "ts", "amt_eur", "from_ccy"))
final_leg = asof_rate(eur_leg, "from_ccy", "USD").withColumn("rn", F.row_number().over(w)).filter("rn = 1")
tri = {r["tx_id"]: round(r["usd"], 2) for r in
       final_leg.select("tx_id", (F.col("amt_eur") * F.col("rate")).alias("usd")).collect()}
print("triangulated INR->EUR->USD:", tri)            # 9000 INR -> 94.5 EUR -> $103.95
assert tri == {104: 103.95}

print("\nTalk track:")
print("* Never store only the converted amount. Fact table carries amount_local + ccy +")
print("  applied_rate + rate_source (deal vs closing) — restatement = re-join, not rewrite.")
print("* Deal-date vs month-end-rate is a BUSINESS decision: revenue uses deal-time rates,")
print("  balance-sheet reporting restates at closing rates — same table, two as-of dates.")
print("* Triangulation risks: path length (rate error compounds per leg), cycle avoidance, and")
print("  consistency — always triangulate through ONE declared base currency.")
print("* As-of join failure mode: rows with no rate yet (future tx, late rate feed) get NULL —")
print("  decide the policy (hold, latest-carry-forward, or reject) explicitly.")
print("\nQ13 OK: effective dates honored (108 vs 165), INR triangulated to 103.95")

## Q14 — Snowflake — Spill Management: Local SSD vs Remote Storage [HARD]

**Scenario (Snowflake performance-ops interview):** a 6TB join ran fine last month, then the cluster key changed and now the same query takes 40 minutes. `QUERY_HISTORY` shows nonzero `BYTES_SPILLED_TO_*`. Explain Snowflake's spill model and drive the fix.

**Requirements**

1. Explain the memory hierarchy: each warehouse size gets a fixed per-node memory + local SSD budget; when a sort/hash build exceeds memory it **spills to local SSD**, and when local fills it spills to **remote storage** — remote spill is dramatically slower, and crossing that boundary is the cliff your query just fell off.
2. Write the spill-hunting query: `SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY` filtered to `BYTES_SPILLED_TO_LOCAL_STORAGE > 0 OR BYTES_SPILLED_TO_REMOTE_STORAGE > 0` over the last week, with `BYTES_SCANNED` for context (illustrative — run on your account).
3. Encode the triage ladder and assert it: remote spill → upsize one step AND cut the scan (zone-map pruning / cluster on the join keys / materialize the pre-agg); local-only spill on a huge scan → pruning first (upsizing to avoid reading data you don't need is waste); no spill + slow → look elsewhere (queueing, locking, skew). Know the non-Snowflake contrast: Spark's `spark.sql.shuffle spill` thresholds from Day 26 are the same physics, different knobs.

In [0]:
# Solution Q14 — spill hunting query + the triage ladder (illustrative, no live Snowflake here)
SPILL_SQL = """
SELECT query_id, warehouse_size, user_name,
       ROUND(total_elapsed_time/1000, 1)                        AS seconds,
       ROUND(bytes_scanned/POWER(1024,3), 1)                    AS scanned_gb,
       ROUND(bytes_spilled_to_local_storage/POWER(1024,2), 1)   AS local_ssd_spill_mb,
       ROUND(bytes_spilled_to_remote_storage/POWER(1024,2), 1) AS remote_storage_spill_mb
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE start_time >= DATEADD(day, -7, CURRENT_TIMESTAMP())
  AND (bytes_spilled_to_local_storage > 0 OR bytes_spilled_to_remote_storage > 0)
ORDER BY total_elapsed_time DESC
LIMIT 50;
"""
print("run on your Snowflake account:\n", SPILL_SQL, sep="")

def spill_triage(scanned_gb, local_spill_mb, remote_spill_mb):
    """Decision ladder from observable symptoms -> action."""
    if remote_spill_mb > 0:
        return ("URGENT: crossed into REMOTE spill — upsize one warehouse step AND cut the"
                " scan: cluster/prune on the join keys or materialize the pre-agg first")
    if local_spill_mb > 0 and scanned_gb > 1000:
        return ("prune first: >1TB scanned with local-only spill — zone-map pruning / cluster"
                " key on join+filter columns; upsizing just to avoid reading data you don't"
                " need is waste")
    if local_spill_mb > 0:
        return ("borderline: single-step upsize or break the join (pre-filter, semi-join,"
                " materialize the build side)")
    return "no spill — look at queueing/locking/skew instead"

cases = {
    "the 6TB join":      spill_triage(6_000, 40_000, 5_200),
    "wide scan, light":  spill_triage(1_800, 300, 0),
    "moderate spill":    spill_triage(80, 4_000, 0),
    "clean but slow":    spill_triage(50, 0, 0),
}
for k, v in cases.items():
    print(f"{k:18s} -> {v}")
assert "REMOTE" in cases["the 6TB join"]
assert "prune" in cases["wide scan, light"]
assert "no spill" in cases["clean but slow"]

print("\nTalk track:")
print("* Physics: warehouse size = more nodes AND more per-node memory+SSD. Local spill (SSD)")
print("  is tolerable; remote spill (object storage) is the cliff — that's where minutes go.")
print("* Prevention beats upsizing: micro-partition pruning (natural zone maps) + clustering")
print("  on the JOIN keys, materialized pre-aggs for repeated heavy builds.")
print("* Same story elsewhere: Spark's shuffle-spill (Day 26) — local disk spill then remote")
print("  if configured; Spark tuning knobs differ, the memory hierarchy doesn't.")
print("* Check the source column: ACCOUNT_USAGE lags (up to ~45 min); INFORMATION_SCHEMA's")
print("  QUERY_HISTORY view is fresher for a live triage.")
print("\nQ14 OK: hunting query + 4-case triage ladder, asserts green")

## Q15 — Snowflake — Streamlit in Snowflake: Governed Data Apps [FAANG]

**Scenario (CTO-level design interview):** a customer wants an interactive app over governed warehouse data — filters, charts, and write-backs. Options: external app + API keys, or **Streamlit in Snowflake (SiS)** — data apps that run INSIDE Snowflake. The interviewer wants to know what that actually means, when it wins, and its limits.

**Requirements**

1. Explain the architecture precisely: app code lives in a **Git repository**, is created/managed in Snowsight, and executes on **Snowflake compute** (warehouse-backed); queries run via a Snowpark session under the **viewer's role** — RBAC, row-access policies, and masking apply to the app exactly as to a worksheet. Data never leaves the governance boundary.
2. Write the app skeleton: `st.connection("snowflake")` → Snowpark session → `session.table(...)`, `session.sql(...)`; Streamlit widgets drive parameterized queries. Note the environment constraint: curated Python packages (Anaconda channel) — no arbitrary pip installs.
3. Build the decision matrix (runnable): given app requirements (governed warehouse data only? external APIs/SDKs? GPU/custom runtime? multi-cloud lakehouse?), decide SiS vs Databricks Apps vs external service — and assert the verdicts. Know the Databricks analogue: Databricks Apps hosts Streamlit/Flask apps against the lakehouse with OAuth + UC governance — same concept, different data plane.

In [0]:
# Solution Q15 — app skeleton (illustrative) + the serve-where decision matrix
APP_SKELETON = '''
# app.py — runs INSIDE Snowflake (Streamlit in Snowflake)
import streamlit as st

# Session runs under the VIEWER's role: RBAC, row policies, masking all apply
conn = st.connection("snowflake")          # Snowpark-backed session
session = conn.session()

region = st.selectbox("Region", ["APAC", "EMEA", "AMER"])
df = (session.table("ANALYTICS.ORDERS")
      .filter(col("REGION") == region)
      .group_by(col("CUSTOMER_SEGMENT"))
      .agg(sum_(col("AMOUNT_USD")).alias("REV")))
st.bar_chart(df.to_pandas(), x="CUSTOMER_SEGMENT", y="REV")
'''
print(APP_SKELETON)

def serve_where(needs_external_apis, needs_gpu_or_custom_runtime, needs_governed_warehouse_only,
                lakehouse_not_snowflake):
    """SiS vs Databricks Apps vs external service — verdict + reason."""
    if lakehouse_not_snowflake:
        return ("Databricks Apps", "governed app over the LAKEHOUSE — same pattern as SiS,"
                " Unity Catalog + OAuth instead of Snowflake roles")
    if needs_gpu_or_custom_runtime or needs_external_apis:
        return ("external service", "SiS runs curated packages inside Snowflake compute only —"
                " external APIs/GPU/custom runtimes need their own compute plane")
    if needs_governed_warehouse_only:
        return ("Streamlit in Snowflake", "data never leaves Snowflake: viewer-role RBAC, row"
                " access policies, masking apply to the app exactly as to a worksheet")
    return ("judgment call", "SiS for speed-to-market on governed data; external when the app")

cases = {
    "sales ops dashboard":   serve_where(False, False, True, False),
    "app calling Stripe API": serve_where(True,  False, True, False),
    "lakehouse ML demo":     serve_where(False, False, True, True),
}
for k, (where, why) in cases.items():
    print(f"{k:24s} -> {where}: {why[:70]}...")
assert cases["sales ops dashboard"][0] == "Streamlit in Snowflake"
assert cases["app calling Stripe API"][0] == "external service"
assert cases["lakehouse ML demo"][0] == "Databricks Apps"

print("\nTalk track:")
print("* The governance sentence is the interview winner: queries execute under the VIEWER's")
print("  role on Snowflake compute — the app cannot show data the user couldn't see anyway.")
print("* Ops story: Git-backed source control, Snowsight lifecycle, warehouse-backed billing;")
print("  environment = curated Anaconda packages — plan your imports before pitching.")
print("* Databricks parallel: Databricks Apps = Streamlit/Flask against the lakehouse with UC")
print("  governance + OAuth — pick the plane where the DATA lives, not the vendor you like.")
print("\nQ15 OK: skeleton + 3/3 verdicts on the decision matrix")

## Q16 — Monitoring — StreamingQueryListener & the Progress JSON Contract [FAANG]

**Scenario (production SRE interview):** your Structured Streaming job's micro-batch latency crept from 2s to 90s over a week and nobody noticed — because "the query was up" was the only check. The interviewer wants the **programmatic observability contract**: what events Spark emits, what's in the progress JSON, and how you'd build lag/throughput alerts on it.

**Requirements**

1. Build a real (mini) streaming query — parquet source, explicit schema, memory sink, `availableNow` trigger (serverless: `processingTime`/continuous triggers raise `INFINITE_STREAMING_TRIGGER_NOT_SUPPORTED`), checkpointed.
2. Read `query.recentProgress[0]` and verify the contract fields: `batchId`, `durationMs` (getBatch / queryPlanning / latestOffset / walCommit / addBatch — the per-phase latency breakdown), `sources`, `stateOperators`. Assert each is present with real values.
3. Register a **`StreamingQueryListener`** (works on Spark Connect) with `onQueryStarted` / `onQueryProgress` / `onQueryTerminated` handlers — the production pattern: push each progress event to metrics/alerting, alert on batch duration SLO and source backlog, and treat `onQueryTerminated` with exception as a page. Say the Connect caveat we validated on Day 31: `numInputRows` can be `None` in progress records — don't build alerts on it.

In [0]:
# Solution Q16 — real mini-stream + progress JSON verification + listener wiring
src16 = f"{BASE}/q16_src"; os.makedirs(src16, exist_ok=True)
for i in range(2):
    pd.DataFrame({"v": [i, i * 2]}).to_parquet(f"{src16}/p{i}.parquet")
    time.sleep(0.05)
schema16 = T.StructType([T.StructField("v", T.LongType())])
qn16 = f"q16_{uuid.uuid4().hex[:4]}"; ck16 = f"{BASE}/q16_ckpt"
q16 = (spark.readStream.schema(schema16).parquet(src16)
       .writeStream.format("memory").queryName(qn16)
       .option("checkpointLocation", ck16)
       .trigger(availableNow=True).start())     # availableNow: serverless-safe one-shot
q16.awaitTermination()

n_rows = spark.sql(f"SELECT COUNT(*) n FROM {qn16}").collect()[0]["n"]
prog = q16.recentProgress[0]
phases = sorted((prog.get("durationMs") or {}).keys())
print(f"rows landed: {n_rows} | batchId: {prog.get('batchId')}")
print("durationMs phases:", phases)
print("sources:", len(prog.get("sources") or []),
      "| stateOperators:", len(prog.get("stateOperators") or []))
assert n_rows == 4 and prog.get("batchId") is not None
assert {"getBatch", "queryPlanning", "triggerExecution", "walCommit", "addBatch"} <= set(phases)
assert (prog.get("sources") or [])

# the production pattern: a listener pushes every event to metrics/alerting
from pyspark.sql.streaming import StreamingQueryListener
class PageOnLag(StreamingQueryListener):
    def __init__(self): self.events = []
    def onQueryStarted(self, e):   self.events.append(("started", e.name, str(e.id)))
    def onQueryProgress(self, e):
        p = e.progress
        self.events.append(("progress", p.get("batchId"),
                            (p.get("durationMs") or {}).get("triggerExecution")))
    def onQueryTerminated(self, e): self.events.append(("terminated", str(e.id)))
listener = PageOnLag()
spark.streams.addListener(listener)      # registered — fires on the NEXT active query
print("listener registered; handlers wired to batch duration metrics")

print("\nTalk track:")
print("* durationMs is the phase-level latency split: getBatch (source fetch), queryPlanning,")
print("  latestOffset, walCommit, addBatch (sink commit) — alert on triggerExecution SLO and")
print("  trend the per-phase split to see WHERE the slowdown lives.")
print("* Connect caveat we validated live: numInputRows can be None in progress records —")
print("  derive throughput from sink counts or source metrics, not that field.")
print("* onQueryTerminated(exception) is your page event; a listener also survives query")
print("  restarts if registered on the session — stream-level monitoring that jobs can't lose.")
print("* Same JSON feeds Genie ZeroOps / system tables monitoring at platform scale — the")
print("  listener is the per-pipeline building block.")
print("\nQ16 OK: 4 rows, contract fields verified, listener wired")

## Q17 — Monitoring — OpenLineage Event Graphs & the Blast Radius Query [HARD]

**Scenario (incident-response interview):** a data quality check fails on `raw_events` at 9am. Which dashboards, models, and downstream tables are now showing wrong numbers — and who owns them? If you can't answer in minutes, you don't have lineage. **OpenLineage** is the open standard for the answer: producer integrations (Spark, dbt, Airflow, Databricks connectors) emit events; you build the graph and query it.

**Requirements**

1. State the event contract: `START`/`COMPLETE`/`FAIL` events per job run; each carries a `job` name and `inputs`/`outputs` datasets (with facets — schema, dataset ownership, run metadata). Producers emit; consumers (e.g. Marquez, DataHub, Databricks Unity Catalog lineage) aggregate.
2. Parse COMPLETE events into a bipartite graph (dataset → job → dataset) and implement the **blast radius** query: from a corrupted dataset, the transitive closure of affected downstream datasets AND the jobs that must re-run. Watch the subtlety we validated live: intermediate datasets (e.g. `silver_events`) ARE part of the blast radius.
3. Assert on a 3-job medallion graph and add the reverse query (upstream debug radius: what feeds this dataset?) as one line of reasoning. Talk: why a standards-based contract beats bespoke lineage tables (multi-tool ecosystems, no vendor lock-in).

In [0]:
# Solution Q17 — parse events, build the graph, query the blast radius
from collections import deque
lineage_events = [json.dumps(e) for e in [
    {"eventType": "COMPLETE", "job": "T1_clean",
     "inputs": ["raw_events"], "outputs": ["silver_events"]},
    {"eventType": "COMPLETE", "job": "T2_rollup",
     "inputs": ["silver_events"], "outputs": ["gold_summary", "gold_daily"]},
    {"eventType": "COMPLETE", "job": "T3_users",
     "inputs": ["silver_events", "raw_users"], "outputs": ["gold_users"]},
]]

consumers, producers = {}, {}          # dataset -> jobs;  job -> output datasets
for raw in lineage_events:
    e = json.loads(raw)                # events arrive as JSON — parse, never trust
    if e["eventType"] != "COMPLETE":
        continue                        # only completed runs contribute real edges
    for i in e["inputs"]:  consumers.setdefault(i, set()).add(e["job"])
    for o in e["outputs"]: producers.setdefault(e["job"], set()).add(o)

def blast_radius(dataset):
    """Transitive downstream closure: affected datasets + jobs that must re-run."""
    seen_ds, seen_jobs, q = {dataset}, set(), deque([dataset])
    while q:
        d = q.popleft()
        for j in consumers.get(d, []):
            if j in seen_jobs: continue
            seen_jobs.add(j)
            for o in producers.get(j, set()):
                if o not in seen_ds:
                    seen_ds.add(o); q.append(o)
    return sorted(seen_ds - {dataset}), sorted(seen_jobs)

affected_ds, affected_jobs = blast_radius("raw_events")
print("raw_events corrupted ->")
print("  affected datasets:", affected_ds)
print("  jobs to re-run:    ", affected_jobs)
assert affected_ds == ["gold_daily", "gold_summary", "gold_users", "silver_events"]
assert affected_jobs == ["T1_clean", "T2_rollup", "T3_users"]

upstream_debug = sorted({i for j in affected_jobs for i in
                         [x for x, js in consumers.items() if j in js]})
print("upstream debug radius (what feeds the affected jobs):", upstream_debug)

print("\nTalk track:")
print("* Blast radius = BFS over the bipartite graph: dataset -> consuming jobs -> their")
print("  outputs. INTERMEDIATE datasets count — silver_events is corrupted too; skipping")
print("  intermediates understates the incident (we asserted the wrong list live, then fixed it).")
print("* Re-run set = affected jobs, ordered by topological sort of the same graph (Day 31's")
print("  Q4 Tarjan condensation guarantees it's a DAG and schedulable).")
print("* Standard contract (START/COMPLETE/FAIL + job/inputs/outputs/facets) is why Spark, dbt,")
print("  and Airflow integrations all compose into ONE graph — no bespoke lineage silos.")
print("* At Databricks scale, Unity Catalog lineage + system tables give you this query built-in;")
print("  OpenLineage is how heterogeneous stacks get the same capability.")
print("\nQ17 OK: 4 datasets + 3 jobs in the blast radius, intermediates included")

## Q18 — Monitoring & Orchestration — Airflow Data Assets vs Databricks Table Triggers [HARD]

**Scenario (orchestration-migration interview):** DAGs chained by cron offsets (`silver at 2:15, gold at 3:00`) break silently whenever a producer runs late — the consumer reads stale data on schedule. Both Airflow and Databricks Jobs now support **data-driven scheduling**: run the consumer when the data is READY, not when the clock says so. Explain the mechanisms and their trade-offs.

**Requirements**

1. Airflow side: **Datasets** (Airflow 2.4+) — tasks declare `outlets=[Dataset("postgres://…/orders")]`, consumer DAGs declare `schedule=[Dataset(...)]`; the consumer triggers when inlets update. Airflow 3 renamed the concept to **Assets** (`Asset` class) — same semantics. Know the constraint: asset scheduling is readiness-gated triggering, not data transfer; backfills re-emit only what you re-run.
2. Databricks side: Jobs can trigger on **table update** (runs when a task/table in UC updates), **file arrival** (cloud storage path), or cron — the same decoupling without an external orchestrator. Say when you still want Airflow: multi-system DAGs, complex cross-orchestrator dependencies, existing operator ecosystem.
3. Simulate (runnable) an asset registry with readiness-gated consumers vs cron-pinned consumers: assert the data-driven consumer never reads stale inlets, while the cron consumer either wastes runs (producer early) or reads stale data (producer late) — quantify both failure modes.

In [0]:
# Solution Q18 — readiness-gated (asset-driven) vs clock-pinned (cron) consumers
AIRFLOW_DAG_EXAMPLE = '''
# Airflow 3: Assets (2.4+ called them Datasets — same semantics, renamed)
from airflow.sdk import Asset

produce_orders = BashOperator(
    task_id="produce_orders", outlets=[Asset("lakehouse://silver/orders")], ...)

gold_dag = DAG("gold", schedule=[Asset("lakehouse://silver/orders"),
                                  Asset("lakehouse://silver/users")], ...)
# gold fires ONLY when BOTH assets updated — never reads a stale inlet
'''
print(AIRFLOW_DAG_EXAMPLE)

class AssetRegistry:
    """Readiness-gated scheduling: consumer fires iff every inlet is newer than its last run."""
    def __init__(self):
        self.updated, self.clock = {}, 0
    def produce(self, asset, ts): self.updated[asset] = ts
    def tick(self, n=1): self.clock += n
    def consumer_ready(self, inlets, last_run):
        return all(self.updated.get(a, -1) > last_run for a in inlets)

INLETS = ["silver/orders", "silver/users"]
reg = AssetRegistry()
runs_asset, runs_cron, stale_reads, wasted_runs = [], [], 0, 0
last_asset_run, last_cron_run = -1, -1
for t in range(6):
    reg.tick()
    if t == 1: reg.produce("silver/orders", t)          # producer 1 lands early
    if t == 4: reg.produce("silver/users", t)            # producer 2 lands LATE
    # cron-pinned consumer: fires every 2 ticks regardless of readiness
    if t % 2 == 1:
        runs_cron.append(t)
        if not reg.consumer_ready(INLETS, last_cron_run): stale_reads += 1
        last_cron_run = t
    # asset-driven consumer: fires only when every inlet is fresh since its last run
    if reg.consumer_ready(INLETS, last_asset_run):
        runs_asset.append(t)
        last_asset_run = t
        if t % 2 == 1: wasted_runs += 0                   # fires when READY, not when free
print(f"asset-driven runs at ticks {runs_asset} | cron runs at {runs_cron}")
print(f"cron consumer: {stale_reads} stale reads, {len([t for t in runs_cron if t < 4 and t >= 1])} wasted")
assert runs_asset == [4]                       # fires ONCE, exactly when both inlets are fresh
assert stale_reads >= 1                        # cron at tick 1 and 3 read stale users table

print("\nTalk track:")
print("* The anti-pattern being replaced: cron offset chains encode LATENCY GUESSES — every")
print("  producer slowdown silently becomes consumer staleness. Asset scheduling encodes")
print("  the DEPENDENCY itself.")
print("* Airflow semantics: outlets/inlets are readiness SIGNALS, not data movement; a dataset")
print("  update event only exists if a task that declares it actually runs — backfills emit")
print("  only what you re-run.")
print("* Databricks native equivalent: Jobs table-update trigger (UC table updates), file")
print("  arrival trigger, cron — same decoupling, one less system; reach for Airflow when the")
print("  DAG spans systems you can't trigger from inside the lakehouse.")
print("* Restraint: assets don't invalidate retries, idempotency, or late-data handling —")
print("  they only fix WHEN. A late producer still needs Day 31's Q10/Q11 machinery.")
print("\nQ18 OK: asset consumer fires exactly at tick 4; cron consumer burns a stale read")

## Day 31 — Self-Assessment Scorecard

Score yourself 1–5 per question **after** attempting it, before reading the solution.

| # | Topic | Tag | Score (1-5) | Confidence (1-5) | Notes |
|---|---|---|---|---|---|
| Q1 | Recursive-CTE shortest path + cycle guard | FAANG | | | |
| Q2 | ROC threshold sweep + AUC rank formula | FAANG | | | |
| Q3 | HyperLogLog scratch (sketch/estimate/merge) | FAANG | | | |
| Q4 | Tarjan SCC pipeline cycle detection | HARD | | | |
| Q5 | Target encoding, k-fold OOF vs leak | HARD | | | |
| Q6 | Distributed ALS via applyInPandas | FAANG | | | |
| Q7 | Task attempts → exclusion → stage retry → speculation | HARD | | | |
| Q8 | Photon native execution + fallback + monitoring | FAANG | | | |
| Q9 | Delta UniForm (Delta ↔ Iceberg readers) | FAANG | | | |
| Q10 | transformWithState TTL semantics | HARD | | | |
| Q11 | Auto Loader discovery modes + cost growth | FAANG | | | |
| Q12 | Mini-dimensions for volatile attributes | HARD | | | |
| Q13 | Multi-currency as-of FX + triangulation | FAANG | | | |
| Q14 | Snowflake spill (local vs remote) triage | HARD | | | |
| Q15 | Streamlit in Snowflake governance | FAANG | | | |
| Q16 | StreamingQueryListener + progress JSON | FAANG | | | |
| Q17 | OpenLineage blast radius | HARD | | | |
| Q18 | Airflow data assets vs table triggers | HARD | | | |

**Day 31 tags: 10 × [FAANG], 8 × [HARD]** — matches the plan.

**Red flags to review tonight** (any of these = re-read that solution):

* Wrote the recursive base case as `cur = src` — or couldn't explain why that double-counts
* Used plain `rank()` for AUC with ties — or `(rank+row_number)/2`
* Stored a count in an HLL register — or couldn't say why merge is element-wise MAX
* Target-encoded from the global total instead of the zip total — or didn't spot the negative leak
* Said UniForm works on path tables, or that UniForm rewrites data
* Forgot that TTL expiry is silent — no callback fires
* Claimed incremental listing is current — it's deprecated; or forgot listing scans the whole prefix every trigger
* Left an auto-remediation-style retry loop unbounded

---

*Day 31 complete — streak: 31/31.*